In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.ar_model import AutoReg
import matplotlib.pyplot as plt
import copy
import json
import random
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# Random seeds
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

In [12]:
# LSTM Config
class Config_LSTM:
    use_bic: bool = False
    data_source = Path('..') / 'Data'
    data_files = [
        'SPY.csv'
    ]
    estimate_type = 'QMLE-Trade'
    value_column = 'Volatility'
    output_root = Path('..') / 'Results'

    lookback_period = 22
    max_bic_lag = 60
    bic_selection_years = 1
    initial_train_years = 5
    validation_years    = 1
    test_months         = 1

    hidden_layers         = [16, 8]
    learning_rate         = 0.001
    batch_size            = 512
    epochs                = 100
    early_stopping_rounds = 10
    n_ensembles           = 10

    lag_candidates = sorted(list(range(1, lookback_period + 1)), reverse=True)
    use_log_transform = True
    target_scaler     = True

    if torch.cuda.is_available():
        device = torch.device('cuda')
        print("Using CUDA")
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
        print("Using MPS")
    else:
        device = torch.device('cpu')
        print("Using CPU")

Using CUDA


In [13]:
class LSTMWithBatchNorm(nn.Module):
    def __init__(self, input_dim: int, hidden_layers, dropout: float = 0.0):
        super().__init__()

        self.input_dim = int(input_dim)
        self.hidden_layers = list(hidden_layers)
        self.dropout = float(dropout)

        # Build LSTM stack
        self.lstm_layers = nn.ModuleList()
        self.lstm_layers.append(nn.LSTM(
            input_size=1,
            hidden_size=self.hidden_layers[0],
            num_layers=1,
            batch_first=True
        ))

        for in_size, out_size in zip(self.hidden_layers[:-1], self.hidden_layers[1:]):
            self.lstm_layers.append(nn.LSTM(
                input_size=in_size,
                hidden_size=out_size,
                num_layers=1,
                batch_first=True
            ))

        self.use_dropout = self.dropout > 0.0
        if self.use_dropout:
            self.dropout_layer = nn.Dropout(self.dropout)

        last_hidden = self.hidden_layers[-1]
        self.bn = nn.BatchNorm1d(last_hidden)
        self.fc_out = nn.Linear(last_hidden, 1)

        nn.init.kaiming_uniform_(self.fc_out.weight, a=np.sqrt(5))
        if self.fc_out.bias is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.fc_out.weight)
            bound = 1 / np.sqrt(fan_in)
            nn.init.uniform_(self.fc_out.bias, -bound, bound)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(-1)
        elif x.dim() == 3:
            pass
        else:
            raise ValueError(f"Expected input of shape (N, L) or (N, L, C), got {tuple(x.shape)}")

        for lstm in self.lstm_layers:
            x, _ = lstm(x)

        last = x[:, -1, :]
        last = self.bn(last)
        if self.use_dropout:
            last = self.dropout_layer(last)

        out = self.fc_out(last)
        return out

In [14]:
# Data loading and panel construction
def load_dacheng_xiu_panel(
    data_directory,
    data_files,
    estimate_type='QMLE-Trade',
    value_column='Volatility'
):
    data_directory = Path(data_directory)
    if not data_directory.is_dir():
        raise FileNotFoundError(f"Data directory does not exist: {data_directory}")
    if not data_files:
        raise ValueError("Add at least one Dacheng Xiu CSV filename to data_files")

    csv_paths = []
    for filename in data_files:
        filename_path = Path(filename)
        if filename_path.suffix == '':
            filename_path = filename_path.with_suffix('.csv')
        csv_path = data_directory / filename_path
        if not csv_path.is_file():
            raise FileNotFoundError(f"Selected CSV does not exist: {csv_path}")
        csv_paths.append(csv_path)

    if len(set(csv_paths)) != len(csv_paths):
        raise ValueError("data_files contains duplicate filenames")

    required_columns = {'Symbol', 'PN', 'Type', 'Date', value_column}
    selected_frames = []

    for csv_path in csv_paths:
        source_data = pd.read_csv(csv_path)
        missing_columns = required_columns.difference(source_data.columns)
        if missing_columns:
            raise ValueError(
                f"{csv_path.name} is missing columns: {sorted(missing_columns)}"
            )

        ticker_data = source_data.loc[
            source_data['Type'].eq(estimate_type),
            ['Symbol', 'Date', value_column]
        ].copy()
        if ticker_data.empty:
            raise ValueError(
                f"{csv_path.name} contains no rows for estimate type '{estimate_type}'"
            )

        ticker_data['Date'] = pd.to_datetime(ticker_data['Date'], errors='raise')
        ticker_data[value_column] = pd.to_numeric(
            ticker_data[value_column], errors='raise'
        )
        selected_frames.append(ticker_data)

    combined_data = pd.concat(selected_frames, ignore_index=True)
    duplicate_mask = combined_data.duplicated(['Date', 'Symbol'], keep=False)
    if duplicate_mask.any():
        duplicate_examples = combined_data.loc[
            duplicate_mask, ['Date', 'Symbol']
        ].drop_duplicates().head(5)
        raise ValueError(
            "Multiple selected observations found for the same symbol/date. "
            f"Examples:\n{duplicate_examples.to_string(index=False)}"
        )

    panel = combined_data.pivot(
        index='Date', columns='Symbol', values=value_column
    ).sort_index()
    panel.index.name = 'date'
    panel.columns.name = None

    print(
        f"Constructed panel from {len(csv_paths)} Dacheng Xiu CSV file(s) "
        f"using {estimate_type}; {value_column} values were not squared"
    )
    print(f"Panel shape: {panel.shape[0]} dates x {panel.shape[1]} symbols")
    print(f"Global date range: {panel.index.min().date()} to {panel.index.max().date()}")
    return panel


def get_stock_series(rv_df, ticker):
    s = rv_df[ticker]
    # Identify the stock's own start/end (first/last non-NaN)
    valid = s.dropna()
    if len(valid) == 0:
        raise ValueError(f"No data for ticker {ticker}")
    stock_start = valid.index[0]
    stock_end   = valid.index[-1]
    print(f"  [{ticker}] Data: {stock_start.date()} to {stock_end.date()} ({len(valid)} obs)")
    return s  # keep NaNs for global alignment

def create_lagged_features(series, lags, use_log_transform=True):
    df = pd.DataFrame({'RV_daily': series})
    df['log_RV'] = np.log(df['RV_daily'].clip(lower=1e-8))
    feature_source = df['log_RV'] if use_log_transform else df['RV_daily']
    for lag in lags:
        df[f'lag_{lag}'] = feature_source.shift(lag)
    df = df.dropna()
    return df

In [15]:
# BIC lag selection
def calculate_BIC(selection_data, max_lag: int = 60):
    bic_values = {}
    for lag in range(1, max_lag + 1):
        model  = AutoReg(
            selection_data, lags=lag, old_names=False, hold_back=max_lag
        )
        result = model.fit()
        bic_values[lag] = result.bic

    best_lag = min(bic_values, key=bic_values.get)

    lookback = list(range(best_lag, 0, -1))
    print(f"    BIC-selected lookback period: {best_lag} ({len(lookback)} lags)")
    return lookback, best_lag

In [16]:
# Training utilities
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_loss  = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter   = 0

def train_single_model(X_train, y_train, X_val, y_val, config, verbose=False):
    X_train_tensor = torch.FloatTensor(X_train).to(config.device)
    y_train_tensor = torch.FloatTensor(y_train).to(config.device)
    X_val_tensor   = torch.FloatTensor(X_val).to(config.device)
    y_val_tensor   = torch.FloatTensor(y_val).to(config.device)

    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader  = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)

    model     = LSTMWithBatchNorm(X_train.shape[1], config.hidden_layers).to(config.device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)

    early_stopping = EarlyStopping(patience=config.early_stopping_rounds)
    best_val   = float('inf')
    best_state = None
    best_epoch = 0

    for epoch in range(config.epochs):
        model.train()
        epoch_train_loss = 0.0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            out  = model(batch_X)
            loss = criterion(out, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_train_loss += loss.item()

        avg_train_loss = epoch_train_loss / max(1, len(train_loader))

        model.eval()
        with torch.inference_mode():
            vout  = model(X_val_tensor)
            vloss = criterion(vout, y_val_tensor).item()

        if (best_val - vloss) > early_stopping.min_delta:
            best_val   = vloss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1

        early_stopping(vloss)
        if early_stopping.early_stop:
            if verbose:
                print(f"Early stopping at epoch {epoch+1}; best={best_epoch} val={best_val:.6f}")
            break

        if verbose and (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{config.epochs}  Train: {avg_train_loss:.6f}  Val: {vloss:.6f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

def set_seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def train_ensemble(X_train, y_train, X_val, y_val, config):
    models    = []
    base_seed = 42
    print(f"    Training ensemble of {config.n_ensembles} models...")
    for i in range(config.n_ensembles):
        set_seed_all(base_seed + i)
        model = train_single_model(X_train, y_train, X_val, y_val, config, verbose=False)
        models.append(model)
    return models

def predict_ensemble(models, X, config):
    X_tensor    = torch.FloatTensor(X).to(config.device)
    predictions = []
    for model in models:
        model.eval()
        with torch.no_grad():
            predictions.append(model(X_tensor).cpu().numpy())
    return np.stack(predictions, axis=0)

In [17]:
# Metrics
def calculate_mse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred)

def calculate_qlike(y_true, y_pred):
    y_true = np.maximum(y_true, 1e-8)
    y_pred = np.maximum(y_pred, 1e-8)
    return np.mean(y_true / y_pred - np.log(y_true / y_pred) - 1)

In [18]:
# Per-stock expanding window forecast
def expanding_window_forecast_single(
    ticker: str,
    rv_series: pd.Series,   # raw RV values
    global_index: pd.DatetimeIndex,
    config
):
    # build lagged feature frame for this stock
    stock_rv = rv_series.dropna()          # restrict to the stock's own date range

    if config.use_bic:
        df_with_lags = create_lagged_features(
            stock_rv,
            list(range(1, config.max_bic_lag + 1)),
            use_log_transform=config.use_log_transform
        )
    else:
        df_with_lags = create_lagged_features(
            stock_rv,
            config.lag_candidates,
            use_log_transform=config.use_log_transform
        )
        feature_cols = [col for col in df_with_lags.columns if col.startswith('lag_')]

    # expanding window setup
    start_date        = df_with_lags.index[0]
    initial_train_end = start_date + pd.DateOffset(years=config.initial_train_years)
    current_test_start = initial_train_end + pd.DateOffset(years=config.validation_years)

    # Storage: one value per global date
    pred_dict = {}   # date -> predicted RV
    bic_dict  = {}   # date -> chosen lookback period (int)

    window_count = 0

    while current_test_start <= df_with_lags.index[-1]:
        window_count += 1

        train_end = current_test_start - pd.DateOffset(years=config.validation_years)
        next_test_start = current_test_start + pd.DateOffset(months=config.test_months)

        # Half-open validation and test windows prevent boundary overlap:
        # validation = (train_end, current_test_start)
        # test       = [current_test_start, next_test_start)
        train_data = df_with_lags.loc[
            (df_with_lags.index >= start_date)
            & (df_with_lags.index <= train_end)
        ]
        val_data = df_with_lags.loc[
            (df_with_lags.index > train_end)
            & (df_with_lags.index < current_test_start)
        ]
        test_data = df_with_lags.loc[
            (df_with_lags.index >= current_test_start)
            & (df_with_lags.index < next_test_start)
        ]

        # Keep memory selection explicit. This dedicated slice covers the most
        # recent configured number of years before the nominal test start.
        bic_selection_start = (
            current_test_start - pd.DateOffset(years=config.bic_selection_years)
        )
        bic_selection_data = df_with_lags.loc[
            (df_with_lags.index >= bic_selection_start)
            & (df_with_lags.index < current_test_start)
        ].copy()

        if (len(test_data) == 0 or len(train_data) == 0 or len(val_data) == 0
                or (config.use_bic and len(bic_selection_data) == 0)):
            break

        print(f"  [{ticker}] Window {window_count}: "
              f"train {train_data.index[0].date()}–{train_data.index[-1].date()} "
              f"| val {val_data.index[0].date()}–{val_data.index[-1].date()} "
              f"| test {test_data.index[0].date()}–{test_data.index[-1].date()} "
              f"({len(test_data)} obs)")

        # lag selection
        if config.use_bic:
            bic_lags, best_lag_int = calculate_BIC(
                bic_selection_data['log_RV'], config.max_bic_lag
            )
            current_feature_cols   = [f'lag_{lag}' for lag in bic_lags]
        else:
            current_feature_cols = feature_cols
            best_lag_int         = None

        # features & targets
        X_train = train_data[current_feature_cols].values
        X_val   = val_data[current_feature_cols].values
        X_test  = test_data[current_feature_cols].values

        y_train_original = train_data['RV_daily'].values.reshape(-1, 1)
        y_val_original   = val_data['RV_daily'].values.reshape(-1, 1)
        y_test_original  = test_data['RV_daily'].values.reshape(-1, 1)

        if config.use_log_transform:
            y_train = np.log(np.maximum(y_train_original, 1e-8))
            y_val   = np.log(np.maximum(y_val_original,   1e-8))
        else:
            y_train = y_train_original.copy()
            y_val   = y_val_original.copy()

        # scaling
        feature_scaler  = StandardScaler()
        X_train_scaled  = feature_scaler.fit_transform(X_train)
        X_val_scaled    = feature_scaler.transform(X_val)
        X_test_scaled   = feature_scaler.transform(X_test)

        if config.target_scaler:
            target_scaler   = StandardScaler()
            y_train_scaled  = target_scaler.fit_transform(y_train)
            y_val_scaled    = target_scaler.transform(y_val)
        else:
            y_train_scaled  = y_train
            y_val_scaled    = y_val
            target_scaler   = None

        # train & predict
        models = train_ensemble(X_train_scaled, y_train_scaled, X_val_scaled, y_val_scaled, config)

        test_pred_scaled_all = predict_ensemble(models, X_test_scaled, config)

        if config.target_scaler and target_scaler is not None:
            test_log_all = np.stack(
                [target_scaler.inverse_transform(p) for p in test_pred_scaled_all], axis=0
            )
        else:
            test_log_all = test_pred_scaled_all

        if config.use_log_transform:
            test_pred = np.mean(np.exp(test_log_all), axis=0)
        else:
            test_pred = np.mean(test_log_all, axis=0)

        test_pred = np.maximum(test_pred, 1e-12).flatten()

        # store predictions aligned to global index
        for i, date in enumerate(test_data.index):
            pred_dict[date] = test_pred[i]
            if config.use_bic:
                bic_dict[date] = best_lag_int   # same lookback chosen for whole test window

        current_test_start = next_test_start

    # reindex to global calendar
    pred_series = pd.Series(pred_dict, name=ticker).reindex(global_index)
    bic_series  = pd.Series(bic_dict,  name=ticker).reindex(global_index)

    return pred_series, bic_series

In [19]:
# Run all stocks (sequential, GPU-friendly)

def run_all_stocks(rv_df: pd.DataFrame, config) -> tuple:
    global_index = rv_df.index
    tickers      = rv_df.columns.tolist()

    pred_cols = {}
    bic_cols  = {}

    for ticker in tickers:
        print(f"\n{'='*60}")
        print(f"Processing: {ticker}")
        print(f"{'='*60}")
        try:
            pred_s, bic_s = expanding_window_forecast_single(
                ticker      = ticker,
                rv_series   = rv_df[ticker],
                global_index= global_index,
                config      = config
            )
            pred_cols[ticker] = pred_s
            bic_cols[ticker]  = bic_s
        except Exception as e:
            print(f"  [{ticker}] FAILED: {e}")
            pred_cols[ticker] = pd.Series(np.nan, index=global_index, name=ticker)
            bic_cols[ticker]  = pd.Series(np.nan, index=global_index, name=ticker)

    pred_df = pd.DataFrame(pred_cols, index=global_index)
    bic_df  = pd.DataFrame(bic_cols,  index=global_index)

    return pred_df, bic_df

In [20]:
# Main entry point

def save_lstm_outputs(pred_df, bic_df, input_df, config):
    if config.use_bic:
        model_tag = 'lstm_bic'
        output_dir = Path(config.output_root) / 'LSTM_BIC'
    else:
        model_tag = f'lstm_{config.lookback_period}'
        output_dir = Path(config.output_root) / f'LSTM_{config.lookback_period}'

    output_dir.mkdir(parents=True, exist_ok=True)

    prediction_path = output_dir / f'predictions_{model_tag}.csv'
    pred_df.to_csv(prediction_path, index_label='date')

    output_paths = {'predictions': str(prediction_path)}
    if config.use_bic:
        lookback_path = output_dir / f'lookback_periods_{model_tag}.csv'
        bic_df.to_csv(lookback_path, index_label='date')
        output_paths['lookback_periods'] = str(lookback_path)

    non_nan_predictions = int(pred_df.notna().sum().sum())
    run_config = {
        'model': model_tag,
        'generated_at_utc': pd.Timestamp.now(tz='UTC').isoformat(),
        'use_bic': bool(config.use_bic),
        'fixed_lookback_period': None if config.use_bic else int(config.lookback_period),
        'max_bic_lag': int(config.max_bic_lag),
        'bic_selection_years': int(config.bic_selection_years),
        'initial_train_years': int(config.initial_train_years),
        'validation_years': int(config.validation_years),
        'test_months': int(config.test_months),
        'hidden_layers': list(config.hidden_layers),
        'learning_rate': float(config.learning_rate),
        'batch_size': int(config.batch_size),
        'epochs': int(config.epochs),
        'early_stopping_rounds': int(config.early_stopping_rounds),
        'n_ensembles': int(config.n_ensembles),
        'use_log_transform': bool(config.use_log_transform),
        'target_scaler': bool(config.target_scaler),
        'estimate_type': config.estimate_type,
        'value_column': config.value_column,
        'data_source': str(config.data_source),
        'data_files': list(config.data_files),
        'load_all_csv_files': not bool(config.data_files),
        'input_start_date': input_df.index.min().isoformat(),
        'input_end_date': input_df.index.max().isoformat(),
        'input_dates': int(input_df.shape[0]),
        'symbols': input_df.columns.tolist(),
        'symbol_count': int(input_df.shape[1]),
        'prediction_shape': list(pred_df.shape),
        'non_nan_predictions': non_nan_predictions,
        'device': str(config.device),
        'output_files': output_paths,
    }

    config_path = output_dir / f'run_config_{model_tag}.json'
    output_paths['run_config'] = str(config_path)
    with config_path.open('w', encoding='utf-8') as config_file:
        json.dump(run_config, config_file, indent=2)

    return output_dir, output_paths

def main_lstm():
    print("="*60)
    print("LSTM NEURAL NETWORK – ALL STOCKS")
    print("="*60)

    config = Config_LSTM()
    print(f"\nUsing device: {config.device}")
    print(f"BIC enabled:  {config.use_bic}")

    # Load data
    print(f"\nLoading data from: {config.data_source}")
    rv_df = load_dacheng_xiu_panel(
        config.data_source,
        config.data_files,
        estimate_type=config.estimate_type,
        value_column=config.value_column
    )

    # Run forecasts
    pred_df, bic_df = run_all_stocks(rv_df, config)

    # Save output
    output_dir, output_paths = save_lstm_outputs(pred_df, bic_df, rv_df, config)
    print(f"\nSaved model outputs to: {output_dir}")
    for output_name, output_path in output_paths.items():
        print(f"  {output_name}: {output_path}")

    print(f"\nOutput shape: {pred_df.shape[0]} dates x {pred_df.shape[1]} stocks")
    non_nan = pred_df.notna().sum().sum()
    print(f"Total predictions made: {non_nan:,}")

    return pred_df, bic_df

if __name__ == '__main__':
    pred_df, bic_df = main_lstm()

LSTM NEURAL NETWORK – ALL STOCKS

Using device: cuda
BIC enabled:  False

Loading data from: ..\Data
Constructed panel from 1 Dacheng Xiu CSV file(s) using QMLE-Trade; Volatility values were not squared
Panel shape: 7301 dates x 1 symbols
Global date range: 1996-01-02 to 2025-04-29

Processing: SPY
  [SPY] Window 1: train 1996-02-02–2001-02-02 | val 2001-02-03–2002-02-03 | test 2002-02-04–2002-03-04 (20 obs)
    Training ensemble of 10 models...
  [SPY] Window 2: train 1996-02-02–2001-03-02 | val 2001-03-03–2002-03-03 | test 2002-03-04–2002-04-04 (23 obs)
    Training ensemble of 10 models...
  [SPY] Window 3: train 1996-02-02–2001-04-02 | val 2001-04-03–2002-04-03 | test 2002-04-04–2002-05-04 (22 obs)
    Training ensemble of 10 models...
  [SPY] Window 4: train 1996-02-02–2001-05-02 | val 2001-05-03–2002-05-03 | test 2002-05-04–2002-06-04 (21 obs)
    Training ensemble of 10 models...
  [SPY] Window 5: train 1996-02-02–2001-06-02 | val 2001-06-03–2002-06-03 | test 2002-06-04–2002-07-

KeyboardInterrupt: 